# Dive into time-frequency domain!

In this lecture, you will be guided step by step, to see what is time-frequency (TF) domain, and how to search signal in TF domain.

Here we load the package `pycwb` and `wdm_wavelet`

In [ ]:
import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import pycwb
from wdm_wavelet.wdm import WDM
import matplotlib.pyplot as plt
import numpy as np

## Time domain, frequency domain, and time-frequency domain

In [ ]:
from pycbc.waveform import get_td_waveform, utils, taper_timeseries
from pycbc.types import TimeSeries
from scipy import interpolate

from lal import PC_SI
import json

def BBH_waveform():
    hp, hc = get_td_waveform(approximant="IMRPhenomTPHM",
                         mass1=20,
                         mass2=20,
                         spin1z=0.9,
                         spin2z=0.4,
                         inclination=1.23,
                         coa_phase=2.45,
                         distance=800,
                         delta_t=1.0/2048,
                         f_lower=20)
    hp = utils.taper_timeseries(hp, tapermethod="startend")
    hc = utils.taper_timeseries(hc, tapermethod="startend")
    return hp, hc

def sxs_waveform(sxs_waveform_name):
    # sxs_waveform_name = 'SXS_BBH_4000' #'SXS_BBH_3890'
    sxs_waveform_path = f'../../data/waveforms/{sxs_waveform_name}_Res3.h5'
    with open('../../data/waveforms/metadata.json') as f:
        metadata = json.load(f)

    m1 = 20
    m2 = m1 * metadata[sxs_waveform_name]['q']
    f_lower = np.array(metadata[sxs_waveform_name]['Mlower']) / (m1 + m2)
    print("m1=", m1, "m2=", m2)
    inclination = 0.0
    coa_phase = -np.pi/2
    distance = 800.0

    hp, hc = get_td_waveform(approximant='NR_hdf5',
                                numrel_data=sxs_waveform_path,
                                mass1=m1,
                                mass2=m2,
                                delta_t=1./2048,
                                f_lower = f_lower ,
                                f_ref = 0,
                                inclination=inclination,
                                coa_phase=coa_phase,
                                distance=distance,
                                mode_array= [ [2,-2], [2,-1], [2,1], [2,2], [3,-3], [3, -2], [3,2], [3,3], [4,-4], [4,-3], [4, -2], [4,2], [4,3], [4,4] ],
                            )
    hp = taper_timeseries(hp, tapermethod="TAPER_STARTEND")
    hc = taper_timeseries(hc, tapermethod="TAPER_STARTEND")
    hp.prepend_zeros(512)
    hc.prepend_zeros(512)
    hp.append_zeros(512)
    hc.append_zeros(512)
    
    return hp, hc

def supernova_waveform(waveform_name='s9.0.swbj15.horo.3d'):
    waveform_path = f'../../data/waveforms/all.3D.strains/{waveform_name}.gw.txt'
    ts = np.loadtxt(waveform_path)
    times = ts[:,0]
    hp = ts[:,1]
    hc = ts[:,2]
    distance = 1e3 * PC_SI # 1 kpc to m
    new_time_array = np.arange(times[0], times[-1], 1.0/16384)
    hp_spline = interpolate.splrep(times, hp, k=3, s=0)
    hc_spline = interpolate.splrep(times, hc, k=3, s=0)
    hp_resampled_spline = interpolate.splev(new_time_array, hp_spline, ext=3)
    hc_resampled_spline = interpolate.splev(new_time_array, hc_spline, ext=3)
    hp = TimeSeries(hp_resampled_spline/distance, delta_t=1.0/16384)
    hc = TimeSeries(hc_resampled_spline/distance, delta_t=1.0/16384)
    # hp.append_zeros(1024)
    # hc.append_zeros(1024)
    hp = taper_timeseries(hp, tapermethod="TAPER_STARTEND")
    hc = taper_timeseries(hc, tapermethod="TAPER_STARTEND")
    hp = hp.resample(1.0/2048)
    hc = hc.resample(1.0/2048)
    return hp, hc


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

hp_bbh, hc_bbh = BBH_waveform()
hp_ebbh, hc_ebbh = sxs_waveform('SXS_BBH_4000')
hp_he, hc_he = sxs_waveform('SXS_BBH_3890')
hp_sn, hc_sn = supernova_waveform('s9.0.swbj15.horo.3d')

fig, axs = plt.subplots(1,4, figsize=(14,3))
axs[0].plot(hp_bbh.sample_times, hp_bbh, label='hp')
axs[0].plot(hc_bbh.sample_times, hc_bbh, label='hc')
axs[0].set_title('BBH')
axs[1].plot(hp_ebbh.sample_times, hp_ebbh, label='hp')
axs[1].plot(hc_ebbh.sample_times, hc_ebbh, label='BBH SXS 4000 (hc)')
axs[1].set_title('eBBH')
axs[2].plot(hp_he.sample_times, hp_he, label='BBH SXS 3890 (hp)')
axs[2].plot(hc_he.sample_times, hc_he, label='BBH SXS 3890 (hc)')
axs[2].set_title('Hyperbolic')
axs[3].plot(hp_sn.sample_times, hp_sn, label='Supernova s9.0.swbj15.horo.3d (hp)')
axs[3].plot(hc_sn.sample_times, hc_sn, label='Supernova s9.0.swbj15.horo.3d (hc)')
axs[3].set_title('Supernova')
for ax in axs:
    ax.grid(False)
plt.tight_layout()
plt.show()
plt.close()

In frequency domain

In [ ]:
# convert to frequency domain
fp_bbh = hp_bbh.to_frequencyseries()
fc_bbh = hc_bbh.to_frequencyseries()
fp_ebbh = hp_ebbh.to_frequencyseries()
fc_ebbh = hc_ebbh.to_frequencyseries()
fp_he = hp_he.to_frequencyseries()
fc_he = hc_he.to_frequencyseries()
fp_sn = hp_sn.to_frequencyseries()
fc_sn = hc_sn.to_frequencyseries()

fig, axs = plt.subplots(2,4, figsize=(14,6))
axs[0,0].set_title('BBH')
axs[0,0].plot(hp_bbh.sample_times, hp_bbh, label='hp')
axs[1,0].loglog(fp_bbh.sample_frequencies, abs(fp_bbh), label='hp')

axs[0,1].set_title('eBBH')
axs[0,1].plot(hp_ebbh.sample_times, hp_ebbh, label='hp')
axs[1,1].loglog(fp_ebbh.sample_frequencies, abs(fp_ebbh), label='hp')

axs[0,2].set_title('Hyperbolic')
axs[0,2].plot(hp_he.sample_times, hp_he, label='hp')
axs[1,2].loglog(fp_he.sample_frequencies, abs(fp_he), label='hp')

axs[0,3].set_title('Supernova')
axs[0,3].plot(hp_sn.sample_times, hp_sn, label='hp')
axs[1,3].loglog(fp_sn.sample_frequencies, abs(fp_sn), label='hp')

for ax in axs.flatten():
    ax.grid(False)

plt.tight_layout()
plt.show()
plt.close()

Now, let's move to time-frequency domain

In [ ]:
from wdm_wavelet.wdm import WDM
wdm = WDM(32, 64, 6, 10)
tfmap_bbh_hp = wdm.t2w(hp_bbh)
tfmap_ebbh_hp = wdm.t2w(hp_ebbh)
tfmap_he_hp = wdm.t2w(hp_he)
tfmap_sn_hp = wdm.t2w(hp_sn)


fig, axs = plt.subplots(3,4, figsize=(14,9))
axs[0,0].set_title('BBH')
axs[0,0].plot(hp_bbh.sample_times, hp_bbh, label='hp')
axs[1,0].loglog(fp_bbh.sample_frequencies, abs(fp_bbh), label='hp')
tfmap_bbh_hp.plot_energy(fig=fig, ax=axs[2,0], colorbar=False)

axs[0,1].set_title('eBBH')
axs[0,1].plot(hp_ebbh.sample_times, hp_ebbh, label='hp')
axs[1,1].loglog(fp_ebbh.sample_frequencies, abs(fp_ebbh), label='hp')
tfmap_ebbh_hp.plot_energy(fig=fig, ax=axs[2,1], colorbar=False)

axs[0,2].set_title('Hyperbolic')
axs[0,2].plot(hp_he.sample_times, hp_he, label='hp')
axs[1,2].loglog(fp_he.sample_frequencies, abs(fp_he), label='hp')
tfmap_he_hp.plot_energy(fig=fig, ax=axs[2,2], colorbar=False)

axs[0,3].set_title('Supernova')
axs[0,3].plot(hp_sn.sample_times, hp_sn, label='hp')
axs[1,3].loglog(fp_sn.sample_frequencies, abs(fp_sn), label='hp')
tfmap_sn_hp.plot_energy(fig=fig, ax=axs[2,3], colorbar=False)

for ax in axs.flatten():
    ax.grid(False)

plt.tight_layout()
plt.show()
plt.close()

What if we add the signal into noise? Can we still see it? 

In [ ]:
from pycwb.modules.read_data.mdc import generate_noise

noises = [generate_noise(sample_rate=2048, f_low=16,
                             duration=1200,
                             start_time=-600, seed=seed) for seed in [10, 20]]
noises[0].plot()

In [ ]:
# Project signal to detector
from pycwb.modules.read_data.mdc import project_to_detector
right_ascension = 0
declination = 0
polarization = 0
ifos = ['H1', 'L1']
gps_end_time = 0
hp = hp_ebbh
hc = hc_ebbh

strain = project_to_detector(hp, hc, right_ascension, declination, polarization, ifos, gps_end_time)
strain[0].plot(label='H1')
strain[1].plot(label='L1')
plt.legend()
plt.show()

In [ ]:
# Add signal to noise
noisy_strain = [
    noises[0].add_into(strain[0]), 
    noises[1].add_into(strain[0])
]

noisy_strain[0].plot()
strain[0].plot(label='H1')
strain[1].plot(label='L1')
plt.xlim(-1, 1)

In [ ]:
wdm = WDM(32, 64, 6, 10)
tf_map = wdm.t2w(noisy_strain[0])

fig, ax = plt.subplots(2,1, figsize=(6, 6))
ax[0].plot(noisy_strain[0].sample_times, noisy_strain[0])
ax[0].set_xlim(-1, 0.1)
ax[0].set_ylabel('Strain')
ax[0].grid(False)
tf_map.plot_energy(fig=fig, ax=ax[1], colorbar=False)
ax[1].set_xlim(-1, 1)
ax[1].set_ylim(10, 800)

In [ ]:
from pycwb.config import Config

config = Config()
config.load_from_yaml('user_parameters.yaml')

In [ ]:
from pycwb.modules.data_conditioning.whitening_cwb import whitening_cwb
from pycwb.modules.data_conditioning import data_conditioning

whiten_strains, nRMS = data_conditioning(config, noisy_strain)

In [ ]:
sliced_data = whiten_strains[0].data.time_slice(-2, 2)

wdm = WDM(32, 64, 6, 10)
tf_map = wdm.t2w(sliced_data)

fig, ax = plt.subplots(2,1, figsize=(6, 6))
ax[0].plot(sliced_data.sample_times, sliced_data)
ax[0].set_xlim(-1, 1)
ax[0].set_ylabel('Strain')
ax[0].grid(False)
tf_map.plot_energy(fig=fig, ax=ax[1], colorbar=False)
ax[1].set_xlim(-1, 1)
ax[1].set_ylim(10, 800)